# Part 6 — Perception

Perception turns raw pixels/points into useful world information: objects, surfaces, free space, depth, tracks, and semantic labels.

**Learning style:** mechanisms first → frameworks second → real systems third. The notebook is intentionally slow, explicit, and beginner-friendly.

In [ ]:
# Setup: run this first.
# Works from the repository root. In Colab, clone the repo first, then run from inside it.
from pathlib import Path
import sys, math, random
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    print('Tip: run this notebook from the repository root, or clone the repo in Colab first.')
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

## 1. Mental model

Perception turns raw pixels/points into useful world information: objects, surfaces, free space, depth, tracks, and semantic labels.

Before code, write one sentence in your own words: *what problem does this topic solve?*

## 2. Mechanism and math

Point-cloud perception often starts with filtering, downsampling, plane fitting, and registration. RANSAC estimates a model by repeatedly sampling small subsets and keeping the model with many inliers.

## 3. From-scratch lab

Fit a ground plane and downsample a point cloud using local utilities.

Read every line. The code avoids clever abstractions so you can see the mechanism.

In [ ]:
import numpy as np
from robotics.pointcloud import fit_plane_ransac, voxel_downsample
rng = np.random.default_rng(0)
ground = np.c_[rng.uniform(-2,2,100), rng.uniform(-2,2,100), rng.normal(0,0.01,100)]
object_pts = rng.normal([0,0,1], 0.1, size=(30,3))
cloud = np.vstack([ground, object_pts])
plane, inliers = fit_plane_ransac(cloud, threshold=0.03, iterations=100, seed=0)
down = voxel_downsample(cloud, voxel_size=0.25)
print('plane [a,b,c,d]:', np.round(plane,3))
print('ground inliers:', inliers.sum(), 'of', len(cloud))
print('downsampled:', len(cloud), '->', len(down))

## 3.1 Code reading guide

When you read the previous cell, do not treat it as a black box. Trace it in this order:

1. **Inputs:** what are the given numbers, observations, states, rewards, or measurements?
2. **Internal variables:** what does each variable represent physically or mathematically?
3. **Update rule:** which line is the core mechanism from the math section?
4. **Output:** what should change if the mechanism is working?
5. **Failure case:** what parameter could make the example unstable, wrong, or unsafe?

This habit is the bridge between toy examples and real robotics code: every simulator, ROS node, policy, controller, or perception model still has inputs, state, an update rule, and outputs.

## 4. Framework/practice view

Frameworks: OpenCV for images, Open3D/PCL for point clouds, YOLO/Detectron/MMDetection for object detection, segmentation models for scene understanding.

The goal is not to replace understanding with APIs. The goal is to recognize the same mechanism when a library hides the details.

In [ ]:
try:
    import cv2
    print('OpenCV version:', cv2.__version__)
except ModuleNotFoundError as e:
    print('Install opencv-python for framework practice:', e)

## 4.1 Framework comparison checklist

After running or reading the framework cell, write a small mapping table for yourself:

| Question | Your answer |
|---|---|
| What object/function in the framework replaces the scratch code? |  |
| Which parameters match the math symbols? |  |
| What details does the framework hide? |  |
| What new engineering concerns appear? | installation, devices, logging, data formats, batching, safety, versioning |

This is where top-down learning becomes useful: you learn the professional API **without losing the mechanism**.

## 5. Real-system connection

Perception feeds planners and controllers. A car needs lanes/objects/free space. A robot arm needs object pose. A drone needs obstacles and visual odometry.

## 6. Exercises

1. Move the object closer to the ground. Does RANSAC still separate it?
2. What does a false positive obstacle do to a planner?
3. What does a missed obstacle do?

**Notebook habit:** after each exercise, add a short note explaining what changed and why it matters in a robot/car/drone/VLA stack.